In [1]:
import os
import torch
import torch.nn as nn
import numpy as np
from tqdm.notebook import trange, tqdm
import h5py
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import glob
from torch.utils.data import TensorDataset, random_split, DataLoader
from sklearn.metrics import f1_score, average_precision_score
#from sklearn.utils.class_weight import compute_class_weight    # >>>>>>> not needed if class balancing methods have been used to make a resampled dataset

In [2]:
class CustomCNN(nn.Module):
    def __init__(self,):
        super(CustomCNN, self).__init__()
        self.kernel_size = 3

        # conv layer
        downsample = self._downsample(4096, 1024)
        self.conv1 = nn.Conv1d(in_channels=12,
                               out_channels=32,
                               kernel_size=self.kernel_size,
                               stride=downsample,
                               padding=self._padding(downsample),
                               bias=False)
        self.bn1 = nn.BatchNorm1d(num_features=32)
        #self.max_pool1 = nn.MaxPool1d(kernel_size=2, stride=2)

        downsample_2 = self._downsample(1024, 512)
        self.conv2 = nn.Conv1d(in_channels=32,
                               out_channels=32,
                               kernel_size=self.kernel_size,
                               stride=downsample_2,
                               padding=self._padding(downsample_2),
                               bias=False)
        self.bn2 = nn.BatchNorm1d(num_features=32)

        downsample_3 = self._downsample(512, 256)
        self.conv3 = nn.Conv1d(in_channels=32,
                               out_channels=32,
                               kernel_size=self.kernel_size,
                               stride=downsample_3,
                               padding=self._padding(downsample_3),
                               bias=False)
        self.bn3 = nn.BatchNorm1d(num_features=32)
        
        # ReLU
        self.relu = nn.ReLU()

        # linear layer
        self.lin = nn.Linear(in_features=256*32, out_features=1)
        
        # if needed, add the skip connection code

    def _padding(self, downsample):
        return max(0, int(np.floor((self.kernel_size - downsample + 1) / 2)))

    def _downsample(self, seq_len_in, seq_len_out):
        return int(seq_len_in // seq_len_out)

    def forward(self, x):
        x= x.transpose(2,1)

        x = self.bn1(self.conv1(x))
        x = self.relu(x)
        x = self.bn2(self.conv2(x))
        x = self.relu(x)
        x = self.bn3(self.conv3(x))
        x = self.relu(x)
        x_flat = x.view(x.size(0), -1) # flatten
        x = self.lin(x_flat)

        return x

In [3]:
!pwd

/mimer/NOBACKUP/groups/naiss2024-5-153/Aditya/1-starter-ecg-model


In [4]:
def train_loop(epoch, dataloader, model, optimizer, loss_function, device):
    # model to training mode (important to correctly handle dropout or batchnorm layers)
    model.train()
    # allocation
    total_loss = 0  # accumulated loss
    n_entries = 0   # accumulated number of data points
    # progress bar def
    train_pbar = tqdm(dataloader, desc="Training Epoch {epoch:2d}".format(epoch=epoch), leave=True)
    train_correct = []

    sigmoid = torch.nn.Sigmoid().to(device)
    # training loop
    for traces, diagnoses in train_pbar:
        traces, diagnoses = traces.to(device), diagnoses.to(device)
        
        # data to device (CPU or GPU if available)
        for x,y in dataloader:
            x, y = x.to(device), y.to(device)
            pred = model(x)
            curr_loss = loss_function(pred, y)
            with torch.no_grad():
                train_correct.append((sigmoid(pred).argmax(1) == y).sum().item())

            curr_loss.backward()
            optimizer.step()
            optimizer.zero_grad()

        # Update accumulated values
        total_loss += curr_loss.detach().cpu().numpy()
        n_entries += len(traces)

        # Update progress bar
        train_pbar.set_postfix({'loss': total_loss / n_entries})
    train_pbar.close()
    return total_loss / n_entries

In [5]:
def eval_loop(epoch, dataloader, model, loss_function, device):
    # model to evaluation mode (important to correctly handle dropout or batchnorm layers)
    model.eval()
    # allocation
    total_loss = 0  # accumulated loss
    n_entries = 0   # accumulated number of data points
    avg_precisions = []  # avg precision (pr-auc)
    f1_s = [] # f1-score

    # progress bar def
    eval_pbar = tqdm(dataloader, desc="Evaluation Epoch {epoch:2d}".format(epoch=epoch), leave=True)

    sigmoid = torch.nn.Sigmoid().to(device)
    # evaluation loop
    for traces_cpu, diagnoses_cpu in eval_pbar:
        # data to device (CPU or GPU if available)
        traces, diagnoses = traces_cpu.to(device), diagnoses_cpu.to(device)

        """
        TASK: Insert your code here. This task can be done in 6 lines of code.
        """
        with torch.no_grad():
            for x,y in dataloader:
                xt, yt = x.to(device), y.to(device)
                
                pred = model(xt)
                curr_loss = loss_function(pred, yt)
                
                avg_precisions.append(average_precision_score(yt.cpu(), sigmoid(pred).cpu()))
                f1_s.append(f1_score(yt.cpu(), sigmoid(pred).cpu().argmax(1)))
                # valid_true.append(yt.detach().cpu()) # save all of the true labels here instead

            # Update accumulated values
            total_loss += curr_loss.detach().cpu().numpy()
            n_entries += len(traces)
        # Update progress bar
        eval_pbar.set_postfix({'loss': total_loss / n_entries, 'avg_precision': np.mean(avg_precisions), 'avg_f1_score': np.mean(f1_s)})
    eval_pbar.close()
    return total_loss / n_entries, np.mean(avg_precisions), np.mean(f1_s)

In [6]:
class BatchDataloader:
    def __init__(self, *tensors, bs=1, mask=None):
        nonzero_idx, = np.nonzero(mask)
        self.tensors = tensors
        self.batch_size = bs
        self.mask = mask
        if nonzero_idx.size > 0:
            self.start_idx = min(nonzero_idx)
            self.end_idx = max(nonzero_idx)
        else:
            self.start_idx = 0
            self.end_idx = 0

    def __next__(self):
        if self.start == self.end_idx:
            raise StopIteration
        end = min(self.start + self.batch_size, self.end_idx)
        batch_mask = self.mask[self.start:end]
        while sum(batch_mask) == 0:
            self.start = end
            end = min(self.start + self.batch_size, self.end_idx)
            batch_mask = self.mask[self.start:end]
        batch = [np.array(t[self.start:end]) for t in self.tensors]
        self.start = end
        self.sum += sum(batch_mask)
        return [torch.tensor(b[batch_mask], dtype=torch.float32) for b in batch]

    def __iter__(self):
        self.start = self.start_idx
        self.sum = 0
        return self

    def __len__(self):
        count = 0
        start = self.start_idx
        while start != self.end_idx:
            end = min(start + self.batch_size, self.end_idx)
            batch_mask = self.mask[start:end]
            if sum(batch_mask) != 0:
                count += 1
            start = end
        return count

In [7]:
# set seed
seed = 42
np.random.seed(seed)
torch.manual_seed(seed)
learning_rate = 1e-4
weight_decay = 1e-1
num_epochs = 10
batch_size = 128

In [9]:
# Set device
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')

tqdm.write("Use device: {device:}\n".format(device=device))
# =============== Build data loaders ======================================#
path_to_h5_train, path_to_csv_train = '../2-soa-ecg-model/data/VDS-code15-upsmpl.hdf5', '../2-soa-ecg-model/data/exams-upsmplCODE15.csv' 

df = pd.read_csv(path_to_csv_train)

tqdm.write("Building data loaders...")
tdataloaders = []
vdataloaders = []

prev = 0
for i, filepath in enumerate(glob.glob("../2-soa-ecg-model/data/code15-12l/*-upsmpl.hdf5")):
    f = h5py.File(filepath, 'r')
    traces = f['tracings']
    print(traces.shape)
    prev = 29000
    labels = torch.tensor(np.array(df)[i*prev:(i+1)*29000]).reshape(-1,1)

    """
    # Train/val split
    dataset = TensorDataset(traces, labels)
    train, valid = random_split(dataset, lengths=[0.7, 0.3])
    print(len(dataset))
    print(len(torch.unique(labels)))
    
    tdataloaders.append(DataLoader(train, batch_size=batch_size, shuffle=True))
    vdataloaders.append(DataLoader(valid, batch_size=batch_size, shuffle=True))
    """
    # Train/ val split
    valid_mask = np.arange(len(labels)) <= int(0.3*len(labels))
    train_mask = ~valid_mask
    
    # Dataloader
    tdataloaders.append(BatchDataloader(traces, labels, bs=batch_size, mask=train_mask))
    vdataloaders.append(BatchDataloader(traces, labels, bs=batch_size, mask=valid_mask))
    
tqdm.write("Done!")

Use device: cuda

Building data loaders...
(29000, 4096, 12)
(29000, 4096, 12)
(29000, 4096, 12)
(29000, 4096, 12)
(29000, 4096, 12)
(29000, 4096, 12)
(29000, 4096, 12)
(8428, 4096, 12)
(29000, 4096, 12)
(29000, 4096, 12)
(29000, 4096, 12)
(29000, 4096, 12)
(29000, 4096, 12)
Done!


In [9]:
# =============== Define model ============================================#
tqdm.write("Define model...")
"""
TASK: Replace the baseline model with your model; Insert your code here
"""
model = CustomCNN()
model.to(device=device)
tqdm.write("Done!\n")

# =============== Define loss function ====================================#
"""
TASK: define the loss; Insert your code here. This can be done in 1 line of code
"""
loss_function = torch.nn.BCEWithLogitsLoss()

# =============== Define optimizer ========================================#
#tqdm.write("Define optimiser...")
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
#tqdm.write("Done!\n")

# =============== Define lr scheduler =====================================#
# TODO advanced students (non mandatory)
"""OPTIONAL: define a learning rate scheduler; Insert your code here"""
lr_scheduler = None #torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=1.0, end_factor=0.25)

# =============== Train model =============================================#
tqdm.write("Training...")
best_loss = np.inf
# allocation
train_loss_all, valid_loss_all = [], []

# MAKE ANOTHER FOR LOOP LOOPING THROUGH ALL DATALOADERS
# loop over epochs
for epoch in trange(1, num_epochs + 1):
    # training loop
    train_loss = train_loop(epoch, train_dataloader, model, optimizer, loss_function, device)
    # validation loop
    valid_loss, avg_precision, avg_roc_auc = eval_loop(epoch, valid_dataloader, model, loss_function, device)

    # collect losses
    train_loss_all.append(train_loss)
    valid_loss_all.append(valid_loss)

    # compute validation metrics for performance evaluation
    """
    TASK: compute validation metrics (e.g. AUROC); Insert your code here
    This can be done e.g. in 5 lines of code
    """
    # y_pred, y_true <-- DO SOMETHING FOR THESE!!

    # save best model: 
    # here we save the model only for the lowest validation loss
    if valid_loss < best_loss:
        # Save model parameters
        torch.save({'model': model.state_dict()}, 'model.pth')
        # Update best validation loss
        best_loss = valid_loss
        # statement
        model_save_state = "Best model -> saved"
    else:
        model_save_state = ""

    # Print message
    tqdm.write('Epoch {epoch:2d}: \t'
                'Train Loss {train_loss:.6f} \t'
                'Valid Loss {valid_loss:.6f} \t'
                '{model_save}'
                .format(epoch=epoch,
                        train_loss=train_loss,
                        valid_loss=valid_loss,
                        model_save=model_save_state))

    # Update learning rate with lr-scheduler
    if lr_scheduler:
        lr_scheduler.step()
    
"""
TASK: Here it can make sense to plot your learning curve; Insert your code here
"""
plt.plot(np.arange(num_epochs), train_loss_all)
plt.plot(np.arange(num_epochs), valid_loss_all)
plt.show()

Define model...
Done!

Training...


  0%|          | 0/10 [00:00<?, ?it/s]

Training Epoch  1:   0%|          | 0/1904 [00:00<?, ?it/s]

Evaluation Epoch  1:   0%|          | 0/816 [00:00<?, ?it/s]

/cephyr/users/adityaa/Alvis/flecg/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:1131: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/cephyr/users/adityaa/Alvis/flecg/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/cephyr/users/adityaa/Alvis/flecg/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:1131: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/cephyr/users/adityaa/Alvis/flecg/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control 

...

/cephyr/users/adityaa/Alvis/flecg/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:1131: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/cephyr/users/adityaa/Alvis/flecg/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/cephyr/users/adityaa/Alvis/flecg/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:1131: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/cephyr/users/adityaa/Alvis/flecg/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control 

Epoch  3: 	Train Loss 0.006876 	Valid Loss 0.001671 	


/cephyr/users/adityaa/Alvis/flecg/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:1131: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/cephyr/users/adityaa/Alvis/flecg/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/cephyr/users/adityaa/Alvis/flecg/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:1131: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/cephyr/users/adityaa/Alvis/flecg/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control 

Training Epoch  4:   0%|          | 0/1904 [00:00<?, ?it/s]

KeyboardInterrupt: 